In [1]:
# K-Means Clustering of Apple Store App Descriptions Using NLP

## 1. Import Libraries

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
```

## 2. Load the Dataset

Make sure `appleStore_description4.csv` is in the same folder as your Jupyter Notebook.

```python
df = pd.read_csv("appleStore_description4.csv")

# Display the first 5 rows
df.head()
```

## 3. Explore the Dataset

```python
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())
```

Check the data types:

```python
df.info()
```

Find the column containing the app descriptions:

```python
for column in df.columns:
    print(column)
```

> **Important:** The code below assumes the description column is called `description`. If your CSV uses a different name, change `description` to the correct column name.

## 4. Remove Missing Descriptions

```python
df = df.dropna(subset=['description']).copy()

df['description'] = df['description'].astype(str)

print("Number of records after removing missing descriptions:", len(df))
```

## 5. NLP Text Preprocessing

We will:

* Convert text to lowercase
* Remove punctuation
* Remove numbers
* Remove stopwords
* Lemmatize words

```python
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Keep only letters and spaces
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # Split into words
    words = text.split()
    
    # Remove stopwords and lemmatize
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words and len(word) > 2
    ]
    
    return ' '.join(words)

df['clean_description'] = df['description'].apply(preprocess_text)

df[['description', 'clean_description']].head()
```

## 6. Convert Text to TF-IDF Features

TF-IDF converts the app descriptions into numerical values that can be used by K-Means.

```python
vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(df['clean_description'])

print("TF-IDF matrix shape:", X.shape)
```

## 7. Find a Suitable Number of Clusters

We can use the **Elbow Method** to examine different values of `k`.

```python
inertia = []

K = range(2, 11)

for k in K:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    kmeans.fit(X)
    inertia.append(kmeans.inertia_)
```

Plot the results:

```python
plt.figure(figsize=(8, 5))

plt.plot(K, inertia, marker='o')

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for Choosing K")

plt.show()
```

Choose a value of `K` based on where the decrease in inertia begins to level off.

For example:

```python
optimal_k = 5
```

You can change `5` based on the elbow plot.

## 8. Train the K-Means Model

```python
kmeans = KMeans(
    n_clusters=optimal_k,
    random_state=42,
    n_init=10
)

df['cluster'] = kmeans.fit_predict(X)

df[['description', 'cluster']].head(10)
```

## 9. Check the Number of Apps in Each Cluster

```python
cluster_counts = df['cluster'].value_counts().sort_index()

print(cluster_counts)
```

Plot the cluster sizes:

```python
plt.figure(figsize=(8, 5))

cluster_counts.plot(kind='bar')

plt.xlabel("Cluster")
plt.ylabel("Number of Apps")
plt.title("Number of Apps in Each Cluster")

plt.xticks(rotation=0)

plt.show()
```

## 10. Evaluate the Clustering Using Silhouette Score

```python
silhouette = silhouette_score(X, df['cluster'])

print("Silhouette Score:", silhouette)
```

A silhouette score closer to 1 generally indicates better separation between clusters, while values around 0 indicate overlapping clusters.

## 11. Find the Most Important Words in Each Cluster

The cluster centres can be used to identify words that are strongly associated with each cluster.

```python
terms = vectorizer.get_feature_names_out()
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]

for cluster_number in range(optimal_k):
    
    print("\n" + "=" * 60)
    print("Cluster", cluster_number)
    print("=" * 60)
    
    top_words = [
        terms[ind]
        for ind in order_centroids[cluster_number, :15]
    ]
    
    print(", ".join(top_words))
```

## 12. Display Example Apps From Each Cluster

```python
for cluster_number in range(optimal_k):
    
    print("\n" + "=" * 70)
    print("CLUSTER", cluster_number)
    print("=" * 70)
    
    cluster_apps = df[df['cluster'] == cluster_number]
    
    for index, row in cluster_apps.head(5).iterrows():
        print("\nApp:", row.get('track_name', 'Unknown'))
        print("Description:", row['description'][:300])
```

If your app-name column has a different name, replace `track_name` with the appropriate column.

## 13. Visualise the Clusters Using PCA

TF-IDF produces thousands of dimensions, so PCA can reduce the data to two dimensions for visualisation.

```python
pca = PCA(n_components=2, random_state=42)

X_pca = pca.fit_transform(X.toarray())

print("PCA shape:", X_pca.shape)
```

Create the plot:

```python
plt.figure(figsize=(10, 7))

scatter = plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=df['cluster'],
    alpha=0.6
)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("K-Means Clustering of Apple Store App Descriptions")

plt.colorbar(scatter, label="Cluster")

plt.show()
```

## 14. Save the Results

Save the original data together with the cleaned descriptions and cluster assignments.

```python
df.to_csv(
    "appleStore_description4_kmeans_results.csv",
    index=False
)

print("Results saved successfully.")
```

## 15. Display the Final Dataset

```python
df.head()
```

## 16. Optional: Automatically Test Different Values of K

This section compares silhouette scores for different numbers of clusters.

```python
silhouette_scores = []

for k in range(2, 11):
    
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    labels = model.fit_predict(X)
    
    score = silhouette_score(X, labels)
    
    silhouette_scores.append(score)
    
    print(f"K = {k}, Silhouette Score = {score:.4f}")
```

Plot the scores:

```python
plt.figure(figsize=(8, 5))

plt.plot(
    range(2, 11),
    silhouette_scores,
    marker='o'
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score for Different K Values")

plt.show()
```

The value of `K` with a relatively high silhouette score can be considered when selecting the number of clusters.

# Complete Workflow

The overall process is:

```text
Apple Store CSV
       ↓
Load Dataset
       ↓
Clean Missing Descriptions
       ↓
NLP Preprocessing
       ↓
TF-IDF Vectorisation
       ↓
Choose K
       ↓
K-Means Clustering
       ↓
Evaluate with Silhouette Score
       ↓
Identify Important Words
       ↓
Visualise Clusters
       ↓
Save Clustered Dataset
```


SyntaxError: invalid character '↓' (U+2193) (3340151032.py, line 365)